![No CAP Project](nocap.png)

_Authored by La’Neyshia Drew, Lucas Carreira, Matthew Konar, Sarah Baeza, and Sierra Mullins_

# Phase Three
**CSCI-P556 Applied Machine Learning · Group 1**

## Phase Leader Plan

**Phase:** Three  
**Phase Leader:**   
**Objectives:** 

## Credit Assignment Plan

| Team Member | Role | Responsibilities |
|---|---|---|
| La'Neyshia Drew |  |  |
| Lucas Carreira |  |  |
| Matthew Konar |  |  |
| Sarah Baeza |  |  |
| Sierra Mullins |  |  |

# Abstract



# Data Lineage



# Evaluation Metrics & Experiments Table



# Load Data and Add Simple Features

In [3]:
import pandas as pd
import numpy as np

DATA_DIR = "/root/shared/group1-hcdr-no-cap/data"

train = pd.read_csv(DATA_DIR + "/application_train.csv")
test = pd.read_csv(DATA_DIR + "/application_test.csv")

print(train.shape, test.shape)

# fix strange value in DAYS_EMPLOYED from phase 2
train["DAYS_EMPLOYED"] = train["DAYS_EMPLOYED"].replace(365243, np.nan)
test["DAYS_EMPLOYED"] = test["DAYS_EMPLOYED"].replace(365243, np.nan)

# ratio features
train["CREDIT_INCOME_RATIO"] = train["AMT_CREDIT"] / train["AMT_INCOME_TOTAL"]
test["CREDIT_INCOME_RATIO"] = test["AMT_CREDIT"] / test["AMT_INCOME_TOTAL"]

train["EXT_MEAN"] = train[["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]].mean(axis=1)
test["EXT_MEAN"] = test[["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]].mean(axis=1)

(307511, 122) (48744, 121)


# RFM Features from Previous Applications and Installments

In [4]:
# previous applications
prev = pd.read_csv(DATA_DIR + "/previous_application.csv")

# group by customer then calculate one number per customer
prev_recency = prev.groupby("SK_ID_CURR")["DAYS_DECISION"].max()    # most recent decision
prev_frequency = prev.groupby("SK_ID_CURR")["SK_ID_PREV"].count()   # number of past applications
prev_monetary = prev.groupby("SK_ID_CURR")["AMT_CREDIT"].mean()     # average credit amount

# put the three results side by side
prev_rfm = pd.DataFrame({
    "PREV_RECENCY": prev_recency,
    "PREV_FREQ": prev_frequency,
    "PREV_MONETARY": prev_monetary
})

# add the new columns to train and test on SK_ID_CURR
train = train.join(prev_rfm, on="SK_ID_CURR")
test = test.join(prev_rfm, on="SK_ID_CURR")

# free the previous applications table (no longer needed)
del prev

# installment payments
ins = pd.read_csv(DATA_DIR + "/installments_payments.csv")

# positive means the payment was made after the due date
ins["LATE_DAYS"] = ins["DAYS_ENTRY_PAYMENT"] - ins["DAYS_INSTALMENT"]

ins_recency = ins.groupby("SK_ID_CURR")["DAYS_INSTALMENT"].max()
ins_frequency = ins.groupby("SK_ID_CURR")["NUM_INSTALMENT_NUMBER"].count()
ins_monetary = ins.groupby("SK_ID_CURR")["AMT_PAYMENT"].mean()
ins_avg_late = ins.groupby("SK_ID_CURR")["LATE_DAYS"].mean()

# free the installments table (about 13.6 million rows, no longer needed)
del ins

ins_rfm = pd.DataFrame({
    "INS_RECENCY": ins_recency,
    "INS_FREQ": ins_frequency,
    "INS_MONETARY": ins_monetary,
    "INS_AVG_LATE": ins_avg_late
})

train = train.join(ins_rfm, on="SK_ID_CURR")
test = test.join(ins_rfm, on="SK_ID_CURR")

print(train.shape, test.shape)

(307511, 131) (48744, 130)


# Preprocess Data and Run Models

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
import time

X = train.drop(columns=["SK_ID_CURR", "TARGET"])
y = train["TARGET"]
X_test = test.drop(columns=["SK_ID_CURR"])

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

num_cols = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()

num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(sparse=False, handle_unknown="ignore"))
])

prep = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols)
])

X_train_p = prep.fit_transform(X_train)
X_val_p = prep.transform(X_val)
X_test_p = prep.transform(X_test)

# Free memory
del X, X_train, X_val, X_test

# experiments log
p3Log = pd.DataFrame(columns=["Experiment", "Features", "Model", "Key hyperparameters", "Val ROC-AUC", "Time (s)"])

def log_result(experiment, features, model, params, auc, secs):
    p3Log.loc[len(p3Log)] = [experiment, features, model, params, round(auc, 4), round(secs, 2)]

# logistic regression
t0 = time.time()
lr = LogisticRegression(C=0.1, class_weight="balanced", max_iter=1000)
lr.fit(X_train_p, y_train)
lr_val_pred = lr.predict_proba(X_val_p)[:, 1]
lr_test_pred = lr.predict_proba(X_test_p)[:, 1]
lr_time = time.time() - t0
log_result("Baseline", "App + ratios + RFM", "LogisticRegression",
           "C=0.1, balanced, max_iter=1000",
           roc_auc_score(y_val, lr_val_pred), lr_time)

# random forest
t0 = time.time()
rf = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1)
rf.fit(X_train_p, y_train)
rf_val_pred = rf.predict_proba(X_val_p)[:, 1]
rf_test_pred = rf.predict_proba(X_test_p)[:, 1]
rf_time = time.time() - t0
log_result("Baseline", "App + ratios + RFM", "RandomForest",
           "100 trees, max_depth=8",
           roc_auc_score(y_val, rf_val_pred), rf_time)

# feature importance from random forest
feat_names = num_cols + list(prep.named_transformers_["cat"]["ohe"].get_feature_names_out(cat_cols))
imp = pd.Series(rf.feature_importances_, index=feat_names).sort_values(ascending=False)
print(imp.head(10))

display(p3Log)

EXT_MEAN                                0.258138
EXT_SOURCE_3                            0.130232
EXT_SOURCE_2                            0.089140
EXT_SOURCE_1                            0.045841
DAYS_EMPLOYED                           0.033799
DAYS_BIRTH                              0.025810
AMT_GOODS_PRICE                         0.019183
NAME_EDUCATION_TYPE_Higher education    0.014254
DAYS_ID_PUBLISH                         0.013604
INS_MONETARY                            0.013389
dtype: float64


,Experiment,Features,Model,Key hyperparameters,Val ROC-AUC,Time (s)
0,Baseline,App + ratios + RFM,LogisticRegression,"C=0.1, balanced, max_iter=1000",0.7515,40.17
1,Baseline,App + ratios + RFM,RandomForest,"100 trees, max_depth=8",0.7418,18.42


# PyTorch Model and Ensemble

In [6]:
import torch
import torch.utils.data
import torch.nn as nn
import torch.optim as optim
import numpy as np

torch.manual_seed(0)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# convert numpy arrays to tensors
X_train_tensor = torch.from_numpy(X_train_p)
X_valid_tensor = torch.from_numpy(X_val_p)
X_test_tensor = torch.from_numpy(X_test_p)

y_train_tensor = torch.from_numpy(y_train.to_numpy())
y_valid_tensor = torch.from_numpy(y_val.to_numpy())

hcdr_train = torch.utils.data.TensorDataset(X_train_tensor, y_train_tensor)
hcdr_valid = torch.utils.data.TensorDataset(X_valid_tensor, y_valid_tensor)

train_batch_size = 96
valid_batch_size = 64

trainloader_hcdr = torch.utils.data.DataLoader(hcdr_train, batch_size=train_batch_size, shuffle=True)
validloader_hcdr = torch.utils.data.DataLoader(hcdr_valid, batch_size=valid_batch_size, shuffle=True)


def run_hcdr_model(
    hidden_layer_neurons=[64, 32],
    opt=optim.Adam,
    epochs=5,
    learning_rate=1e-3
):
    D_in = X_train_p.shape[1]
    D_out = 2

    str_neurons = [str(h) for h in hidden_layer_neurons]
    arch_string = f"{D_in}-{'-'.join(str_neurons)}-{D_out}"

    layers = [
        torch.nn.Linear(D_in, hidden_layer_neurons[0]),
        nn.ReLU(),
    ]

    for i in range(1, len(hidden_layer_neurons)):
        prev, curr = hidden_layer_neurons[i - 1], hidden_layer_neurons[i]
        layers.append(torch.nn.Linear(prev, curr))
        layers.append(nn.ReLU())

    layers.append(torch.nn.Linear(hidden_layer_neurons[-1], D_out))

    model = torch.nn.Sequential(*layers)
    model.to(device)

    loss_fn = nn.CrossEntropyLoss()
    optimizer = opt(model.parameters(), lr=learning_rate)

    def train_epoch(model, loss_fn, optimizer, train_loader):
        running_loss = 0.0
        count = 0
        y_pred = []
        epoch_target = []
        for batch_id, data in enumerate(train_loader):
            inputs, target = data[0].to(device), data[1].to(device)
            optimizer.zero_grad()
            preds = model(inputs.float())
            loss = loss_fn(preds, target)
            loss.backward()
            optimizer.step()
            y_pred.extend(torch.argmax(preds, dim=1).tolist())
            epoch_target.extend(target.tolist())
            running_loss += loss.item()
            count += 1

        loss = np.round(running_loss / count, 3)
        correct = (np.array(y_pred) == np.array(epoch_target))
        accuracy = np.round(correct.sum() / correct.size, 3)
        return loss, accuracy

    def evaluate_model(model, loss_fn, data_loader):
        overall_loss = 0.0
        count = 0
        y_pred = []
        epoch_target = []
        for i, data in enumerate(data_loader):
            inputs, target = data[0].to(device), data[1].to(device)
            preds = model(inputs.float())
            loss = loss_fn(preds, target)
            overall_loss += loss.item()
            y_pred.extend(torch.argmax(preds, dim=1).tolist())
            epoch_target.extend(target.tolist())
            count += 1

        loss = np.round(overall_loss / count, 3)
        correct = (np.array(y_pred) == np.array(epoch_target))
        accuracy = np.round(correct.sum() / correct.size, 3)
        return loss, accuracy

    for epoch in range(epochs):
        train_loss, train_accuracy = train_epoch(model, loss_fn, optimizer, trainloader_hcdr)
        valid_loss, valid_accuracy = evaluate_model(model, loss_fn, validloader_hcdr)
        print(f"Epoch {epoch+1}: Train Accuracy: {train_accuracy}\t Validation Accuracy: {valid_accuracy}")
    print("-" * 50)

    return model, arch_string, train_accuracy, valid_accuracy


# train the MLP
torch.manual_seed(0)

hidden_layer_neurons = [64, 32]
opt = optim.Adam
epochs = 5
learning_rate = 1e-3

t0 = time.time()
model, arch_string, train_accuracy, valid_accuracy = run_hcdr_model(
    hidden_layer_neurons, opt, epochs, learning_rate
)

# probability of class 1 for the AUC and the ensemble
mlp_val_pred = torch.softmax(model(X_valid_tensor.float().to(device)), dim=1)[:, 1].detach().cpu().numpy()
mlp_test_pred = torch.softmax(model(X_test_tensor.float().to(device)), dim=1)[:, 1].detach().cpu().numpy()
mlp_time = time.time() - t0

log_result("Baseline", "App + ratios + RFM", "PyTorch MLP",
           f"{arch_string}, Adam lr={learning_rate}, {epochs} epochs, batch {train_batch_size}",
           roc_auc_score(y_val, mlp_val_pred), mlp_time)

# average ensemble 
ens_val_pred = (lr_val_pred + rf_val_pred + mlp_val_pred) / 3
ens_test_pred = (lr_test_pred + rf_test_pred + mlp_test_pred) / 3
log_result("Baseline", "App + ratios + RFM", "Average Ensemble",
           "mean of LR, RF, MLP",
           roc_auc_score(y_val, ens_val_pred), lr_time + rf_time + mlp_time)

display(p3Log)

# save the Kaggle submission file
sub = pd.DataFrame({
    "SK_ID_CURR": test["SK_ID_CURR"],
    "TARGET": ens_test_pred
})
sub.to_csv("submission_phase3.csv", index=False)
print(sub.head())

Epoch 1: Train Accuracy: 0.918	 Validation Accuracy: 0.919
Epoch 2: Train Accuracy: 0.919	 Validation Accuracy: 0.92
Epoch 3: Train Accuracy: 0.919	 Validation Accuracy: 0.92
Epoch 4: Train Accuracy: 0.92	 Validation Accuracy: 0.919
Epoch 5: Train Accuracy: 0.92	 Validation Accuracy: 0.92
--------------------------------------------------


,Experiment,Features,Model,Key hyperparameters,Val ROC-AUC,Time (s)
0,Baseline,App + ratios + RFM,LogisticRegression,"C=0.1, balanced, max_iter=1000",0.7515,40.17
1,Baseline,App + ratios + RFM,RandomForest,"100 trees, max_depth=8",0.7418,18.42
2,Baseline,App + ratios + RFM,PyTorch MLP,"253-64-32-2, Adam lr=0.001, 5 epochs, batch 96",0.7544,190.00
3,Baseline,App + ratios + RFM,Average Ensemble,"mean of LR, RF, MLP",0.7560,248.60


   SK_ID_CURR    TARGET
0      100001  0.181403
1      100005  0.327445
2      100013  0.131750
3      100028  0.100837
4      100038  0.330599


# Test if RFM features add value or not

In [8]:
rfm_cols = ["PREV_RECENCY", "PREV_FREQ", "PREV_MONETARY",
            "INS_RECENCY", "INS_FREQ", "INS_MONETARY", "INS_AVG_LATE"]

# positions of every column, and of every column except the RFM ones
all_idx = list(range(X_train_p.shape[1]))
keep_idx = [i for i, name in enumerate(feat_names) if name not in rfm_cols]

# 50,000-row sample of the training data
np.random.seed(42)
idx = np.random.choice(X_train_p.shape[0], 50000, replace=False)
Xs, ys = X_train_p[idx], y_train.to_numpy()[idx]

for label, cols in [("App + ratios + RFM (50k sample)", all_idx),
                    ("App + ratios, no RFM (50k sample)", keep_idx)]:
    t0 = time.time()
    m = LogisticRegression(C=0.1, class_weight="balanced", max_iter=1000)
    m.fit(Xs[:, cols], ys)
    auc = roc_auc_score(y_val, m.predict_proba(X_val_p[:, cols])[:, 1])
    log_result("RFM comparison", label, "LogisticRegression",
               "C=0.1, balanced, max_iter=1000", auc, time.time() - t0)

display(p3Log)

,Experiment,Features,Model,Key hyperparameters,Val ROC-AUC,Time (s)
0,Baseline,App + ratios + RFM,LogisticRegression,"C=0.1, balanced, max_iter=1000",0.7515,40.17
1,Baseline,App + ratios + RFM,RandomForest,"100 trees, max_depth=8",0.7418,18.42
2,Baseline,App + ratios + RFM,PyTorch MLP,"253-64-32-2, Adam lr=0.001, 5 epochs, batch 96",0.7544,190.00
3,Baseline,App + ratios + RFM,Average Ensemble,"mean of LR, RF, MLP",0.7560,248.60
4,RFM comparison,App + ratios + RFM (50k sample),LogisticRegression,"C=0.1, balanced, max_iter=1000",0.7458,11.19
5,RFM comparison,"App + ratios, no RFM (50k sample)",LogisticRegression,"C=0.1, balanced, max_iter=1000",0.7434,10.18


# Cross Validation

In [9]:
from sklearn.model_selection import cross_val_score

np.random.seed(42)
idx = np.random.choice(X_train_p.shape[0], 50000, replace=False)
Xs, ys = X_train_p[idx], y_train.to_numpy()[idx]

models = {
    "LogisticRegression": LogisticRegression(C=0.1, class_weight="balanced", max_iter=1000),
    "RandomForest": RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1),
}

rows = []
for name, m in models.items():
    scores = cross_val_score(m, Xs, ys, cv=5, scoring="roc_auc")
    rows.append([name, round(scores.mean(), 4), round(scores.std(), 4)])

cvTable = pd.DataFrame(rows, columns=["Model", "Mean CV ROC-AUC", "Std across 5 folds"])
display(cvTable)

,Model,Mean CV ROC-AUC,Std across 5 folds
0,LogisticRegression,0.7409,0.0078
1,RandomForest,0.7288,0.0073


# Hyperparameter Tuning

## Random Forest

In [10]:
from sklearn.model_selection import GridSearchCV

# random forest grid search
rf_params = {"n_estimators": [100, 200], "max_depth": [6, 8, 12]}
rf_gridsearch = GridSearchCV(RandomForestClassifier(random_state=42, n_jobs=-1),
                             param_grid=rf_params, cv=3, scoring="roc_auc")

print("Performing grid search for random forest...")
t0 = time.time()
rf_gridsearch.fit(Xs, ys)
print("done in %0.3fs" % (time.time() - t0))
print("Best parameters set found:", rf_gridsearch.best_params_)

means = rf_gridsearch.cv_results_["mean_test_score"]
stds = rf_gridsearch.cv_results_["std_test_score"]
for mean, std, params in zip(means, stds, rf_gridsearch.cv_results_["params"]):
    print("%0.4f (+/-%0.04f) for %r" % (mean, std * 2, params))

Performing grid search for random forest...
done in 43.284s
Best parameters set found: {'max_depth': 8, 'n_estimators': 200}
0.7286 (+/-0.0035) for {'max_depth': 6, 'n_estimators': 100}
0.7296 (+/-0.0032) for {'max_depth': 6, 'n_estimators': 200}
0.7288 (+/-0.0046) for {'max_depth': 8, 'n_estimators': 100}
0.7315 (+/-0.0049) for {'max_depth': 8, 'n_estimators': 200}
0.7266 (+/-0.0076) for {'max_depth': 12, 'n_estimators': 100}
0.7308 (+/-0.0086) for {'max_depth': 12, 'n_estimators': 200}


## Logistic Regression

In [11]:
# logistic regression grid search
lr_params = {"C": [0.01, 0.1, 1]}
lr_gridsearch = GridSearchCV(LogisticRegression(class_weight="balanced", max_iter=1000),
                             param_grid=lr_params, cv=3, scoring="roc_auc")

print("Performing grid search for logistic regression...")
t0 = time.time()
lr_gridsearch.fit(Xs, ys)
print("done in %0.3fs" % (time.time() - t0))
print("Best parameters set found:", lr_gridsearch.best_params_)

means = lr_gridsearch.cv_results_["mean_test_score"]
stds = lr_gridsearch.cv_results_["std_test_score"]
for mean, std, params in zip(means, stds, lr_gridsearch.cv_results_["params"]):
    print("%0.4f (+/-%0.04f) for %r" % (mean, std * 2, params))

Performing grid search for logistic regression...
done in 67.379s
Best parameters set found: {'C': 0.01}
0.7413 (+/-0.0072) for {'C': 0.01}
0.7407 (+/-0.0074) for {'C': 0.1}
0.7390 (+/-0.0072) for {'C': 1}


## Retrain

In [12]:
# retrain the best settings on the full training data and log
t0 = time.time()
rf_tuned = RandomForestClassifier(**rf_gridsearch.best_params_, random_state=42, n_jobs=-1)
rf_tuned.fit(X_train_p, y_train)
rf_tuned_val_pred = rf_tuned.predict_proba(X_val_p)[:, 1]
log_result("Tuned", "App + ratios + RFM", "RandomForest",
           str(rf_gridsearch.best_params_),
           roc_auc_score(y_val, rf_tuned_val_pred), time.time() - t0)

t0 = time.time()
lr_tuned = LogisticRegression(C=lr_gridsearch.best_params_["C"], class_weight="balanced", max_iter=1000)
lr_tuned.fit(X_train_p, y_train)
lr_tuned_val_pred = lr_tuned.predict_proba(X_val_p)[:, 1]
log_result("Tuned", "App + ratios + RFM", "LogisticRegression",
           f"C={lr_gridsearch.best_params_['C']}, balanced, max_iter=1000",
           roc_auc_score(y_val, lr_tuned_val_pred), time.time() - t0)

display(p3Log)

,Experiment,Features,Model,Key hyperparameters,Val ROC-AUC,Time (s)
0,Baseline,App + ratios + RFM,LogisticRegression,"C=0.1, balanced, max_iter=1000",0.7515,40.17
1,Baseline,App + ratios + RFM,RandomForest,"100 trees, max_depth=8",0.7418,18.42
2,Baseline,App + ratios + RFM,PyTorch MLP,"253-64-32-2, Adam lr=0.001, 5 epochs, batch 96",0.7544,190.00
3,Baseline,App + ratios + RFM,Average Ensemble,"mean of LR, RF, MLP",0.7560,248.60
4,RFM comparison,App + ratios + RFM (50k sample),LogisticRegression,"C=0.1, balanced, max_iter=1000",0.7458,11.19
5,RFM comparison,"App + ratios, no RFM (50k sample)",LogisticRegression,"C=0.1, balanced, max_iter=1000",0.7434,10.18
6,Tuned,App + ratios + RFM,RandomForest,"{'max_depth': 8, 'n_estimators': 200}",0.7430,27.37
7,Tuned,App + ratios + RFM,LogisticRegression,"C=0.01, balanced, max_iter=1000",0.7520,37.78
